# 单元测试入门 -- 从原理到实战

本笔记结合项目中 `test/` 目录下的真实测试代码，帮助初级开发者从零理解单元测试。

内容包括：
1. **为什么要写测试？** -- 测试的价值与常见借口
2. **pytest 快速上手** -- 基本用法与核心概念
3. **fixture：测试的"前置准备"** -- 理解 yield、scope、conftest
4. **模型层测试** -- 验证数据结构与校验规则
5. **服务层测试** -- 验证业务逻辑
6. **路由层测试** -- 用 TestClient 发 HTTP 请求
7. **Mock：隔离外部依赖** -- 避免连接真实数据库、Redis、下载服务
8. **锁定层测试** -- 并发安全的测试技巧
9. **conftest.py 解读** -- 测试基础设施
10. **实战练习** -- 为 MemoryCache 写测试

---
## 一、为什么要写测试？

先看一段"没测试"的常见场景：

In [ ]:
# 假设你写了这样一个函数
def calculate_discount(price: float, is_vip: bool) -> float:
    if is_vip:
        return price * 0.8
    return price * 0.9

# 某天你重构了一下，改成了这样
def calculate_discount(price: float, is_vip: bool) -> float:
    return price * (0.8 if is_vip else 1.0)  # BUG: 非 VIP 应该是 0.9 不是 1.0

# 如果没人手动测试"非 VIP"的情况，这个 bug 就上线了

### 测试解决什么问题？

| 没有测试 | 有测试 |
|----------|--------|
| 改代码靠"感觉"，怕改坏 | 改完跑测试，几秒知道结果 |
| 回归 bug 靠人来测 | 自动化回归 |
| 新同事不敢改老代码 | 测试就是最好的"说明文档" |
| "在我机器上能跑" | CI 里全跑过才叫能跑 |

### 测试的价值不是"证明代码没 bug"

而是**给你的修改提供一个安全网**。你来改代码，测试来告诉你有没有把什么东西改坏了。

### 常见借口与反驳

| 借口 | 现实 |
|------|------|
| "写测试太慢了" | 不写测试会越来越慢，每次手动验证的时间累积起来远超写测试的时间 |
| "代码很简单不需要测试" | 简单的代码会被改复杂，到时候就没测试保护了 |
| "我手动测过了" | 手动测试每次都要重新做，而且人会累、会忘、会疏忽 |

---
## 二、pytest 快速上手

In [ ]:
# ============================================================
# pytest 最小示例
# ============================================================

import pytest


# --- 基本断言 ---
def test_basic_assert():
    """最基础的测试：assert 表达式为真就通过，为假就失败"""
    result = 1 + 1
    assert result == 2
    assert isinstance(result, int)


# --- 断言异常 ---
def divide(a, b):
    return a / b


def test_raises_exception():
    """验证某段代码会抛出指定的异常"""
    with pytest.raises(ZeroDivisionError):
        divide(1, 0)


# --- 断言异常消息 ---
def test_raises_with_message():
    """验证异常的消息内容"""
    with pytest.raises(ValueError, match="不能为空"):
        raise ValueError("名称不能为空")


# --- 参数化测试：同一逻辑，多组数据 ---
@pytest.mark.parametrize("a,b,expected", [
    (1, 1, 2),
    (0, 0, 0),
    (-1, 1, 0),
    (100, -50, 50),
])
def test_add_parametrized(a, b, expected):
    assert a + b == expected


print("pytest 基础语法就这些：assert、pytest.raises、parametrize")

### pytest 的测试发现规则

pytest 会**自动发现**测试文件，不需要任何注册或配置：

```
pytest 的默认发现规则：
  +-- 文件名以 test_ 开头或 _test 结尾
  +-- 类名以 Test 开头（且不带 __init__）
  +-- 函数名以 test_ 开头
```

### 运行方式

```bash
pytest                          # 运行所有测试
pytest test/test_models.py      # 运行指定文件
pytest -v                       # 显示每个测试的名称和结果
pytest -s                       # 不截获 print 输出
pytest -k "login"               # 只运行名称包含 "login" 的测试
pytest --tb=short               # 简化错误回溯
```

---
## 三、fixture：测试的"前置准备"

fixture 是 pytest 最核心的概念之一。它解决了"每个测试前都要做的准备工作"。

用一个比喻理解：

> **fixture 就像餐厅的后厨**。测试函数（顾客）只需要在参数里写上菜名，pytest（服务员）就会自动把准备好的菜端上来。顾客吃完后（测试结束），服务员负责收拾桌子。

In [ ]:
# ============================================================
# fixture 入门
# ============================================================
import pytest
import os


# 定义一个 fixture：准备测试数据
@pytest.fixture
def sample_user():
    """创建一个测试用的用户数据"""
    user = {"id": 1, "name": "alice", "email": "alice@test.com"}
    return user


def test_fixture_basic(sample_user):
    """测试函数通过参数名获取 fixture 的返回值"""
    assert sample_user["name"] == "alice"
    assert sample_user["id"] == 1


# ============================================================
# fixture 的 setup 和 teardown（通过 yield 实现）
# ============================================================
@pytest.fixture
def temp_file():
    """yield 前是 setup，yield 后是 teardown"""
    # === SETUP ===
    path = "/tmp/test_temp_file.txt"
    with open(path, "w") as f:
        f.write("hello")

    yield path  # 把 path 传给测试

    # === TEARDOWN ===
    os.remove(path)


def test_temp_file(temp_file):
    with open(temp_file) as f:
        assert f.read() == "hello"
    # 测试结束后，临时文件会被 fixture 自动删除


print("fixture 的核心就是：准备 -> yield -> 清理")

### fixture 的 scope（作用域）

| scope | 含义 | 使用场景 |
|-------|------|----------|
| `function`（默认） | 每个测试函数执行一次 | 大多数情况 |
| `class` | 每个测试类执行一次 | 类内多个测试共享数据 |
| `module` | 每个测试文件执行一次 | 启动成本高的资源（如数据库引擎） |
| `session` | 整个测试会话执行一次 | 一次性初始化 |

```python
@pytest.fixture(scope="module")   # 整个文件共享
def expensive_resource():
    print("创建资源（只执行一次）")
    yield "resource"
    print("销毁资源")
```

### conftest.py -- 共享 fixture

项目中 `test/conftest.py` 定义的 fixture 会被**同级和子目录**下所有测试文件自动共享，无需手动 import。

项目中 conftest.py 的 fixture 层级关系：

```
engine (scope=function) -- 创建 SQLite 内存数据库
   +-- session (scope=function) -- 从 engine 获取会话
   +-- client (scope=function) -- 创建 TestClient
         +-- 覆盖 get_session 依赖，使用测试数据库
```

---
## 四、模型层测试 -- 验证数据结构

> 参考文件：`test/test_models.py`

### 测试内容

- Pydantic BaseModel：验证字段是否存在、类型是否正确、默认值、缺少必填字段是否报错
- SQLModel ORM 类：验证实例创建、主键行为

### 为什么模型测试最简单？

因为**模型没有外部依赖**，不需要数据库、不需要网络、不需要文件系统。直接创建一个对象然后 assert 即可。

In [ ]:
# ============================================================
# 模型测试示例（模拟项目中的 APIResponse）
# ============================================================
from typing import Any
from pydantic import BaseModel, ValidationError
import pytest


class APIResponse(BaseModel):
    code: int = 0
    message: str = "success"
    data: Any = None


class TestAPIResponse:
    """APIResponse 模型测试"""

    def test_default_values(self):
        """不传任何参数时，所有字段都有默认值"""
        resp = APIResponse()
        assert resp.code == 0
        assert resp.message == "success"
        assert resp.data is None

    def test_custom_values(self):
        """传入参数后，字段值应该和我们传的一样"""
        resp = APIResponse(code=1, message="error", data={"key": "value"})
        assert resp.code == 1
        assert resp.message == "error"
        assert resp.data == {"key": "value"}

    def test_type_conversion(self):
        """Pydantic 会自动做类型转换"""
        resp = APIResponse(code="123")  # 字符串 "123" -> 整数 123
        assert resp.code == 123
        assert isinstance(resp.code, int)

    def test_wrong_type(self):
        """类型不匹配且无法转换时抛出 ValidationError"""
        with pytest.raises(ValidationError):
            APIResponse(code="abc")  # "abc" 不能转成 int


# 手动验证
tester = TestAPIResponse()
tester.test_default_values()
tester.test_custom_values()
tester.test_type_conversion()
print("模型测试的核心：验证默认值 + 验证自定义值 + 验证校验规则")

### 模型测试的通用模式

```python
class TestYourModel:
    def test_valid(self):
        """正常构造，字段值正确"""
        obj = YourModel(field1=..., field2=...)
        assert obj.field1 == ...

    def test_missing_required(self):
        """缺少必填字段，抛出 ValidationError"""
        with pytest.raises(ValidationError):
            YourModel(field1=...)  # 少了必填的 field2

    def test_default(self):
        """不传可选字段，使用默认值"""
        obj = YourModel()
        assert obj.optional_field == default_value
```

### 项目中测试了哪些模型？

| 模型 | 类型 | 测试要点 |
|------|------|----------|
| `APIResponse` | Pydantic BaseModel | 默认值、自定义值 |
| `UserResponse` | Pydantic BaseModel | 字段赋值 |
| `UserCreateRequest` | Pydantic BaseModel | 正常构造、缺字段 |
| `UserLoginRequest` | Pydantic BaseModel | 正常构造、缺字段 |
| `PasswordUpdateRequest` | Pydantic BaseModel | 正常构造、缺字段 |
| `UserUpdateRequest` | Pydantic BaseModel | 正常构造、缺字段 |
| `UserPublic` | Pydantic BaseModel | 构造、缺字段、无 password |
| `User` (ORM) | SQLModel | 实例创建、id 自动生成 |
| `PdfFileResponse` | Starlette FileResponse | media_type、filename |

---
## 五、服务层测试 -- 验证业务逻辑

> 参考文件：`test/test_services_user.py`

### 与服务层测试的区别

| | 模型层测试 | 服务层测试 |
|---|---|---|
| 依赖 | 无 | 需要数据库 |
| 测试对象 | 数据结构是否正确 | 业务规则是否正确 |
| 典型断言 | `assert obj.field == value` | `assert resp.code == 1` |
| 测试数据 | 直接传参 | 通过 service 函数操作数据 |

### 核心技巧：每个功能都测"成功路径"和"失败路径"

In [ ]:
# ============================================================
# 服务层测试示例（简化版用户服务）
# ============================================================
from dataclasses import dataclass


# --- 模拟数据库 ---
class FakeDB:
    def __init__(self):
        self.users: dict[int, dict] = {}
        self._next_id = 1

    def insert(self, name: str, email: str) -> dict:
        user = {"id": self._next_id, "name": name, "email": email}
        self.users[self._next_id] = user
        self._next_id += 1
        return user

    def find_by_email(self, email: str) -> dict | None:
        for u in self.users.values():
            if u["email"] == email:
                return u
        return None

    def find_by_id(self, user_id: int) -> dict | None:
        return self.users.get(user_id)

    def find_by_name(self, name: str) -> dict | None:
        for u in self.users.values():
            if u["name"] == name:
                return u
        return None

    def update_name(self, user_id: int, new_name: str):
        if user_id in self.users:
            self.users[user_id]["name"] = new_name
            return self.users[user_id]
        return None


# --- 服务层 ---
@dataclass
class RegisterResult:
    code: int = 0
    message: str = "success"
    data: dict | None = None


def register(db: FakeDB, name: str, email: str) -> RegisterResult:
    existing = db.find_by_email(email)
    if existing:
        return RegisterResult(code=1, message="email already registered")
    user = db.insert(name, email)
    return RegisterResult(data=user)


def update_name(db: FakeDB, user_id: int, new_name: str) -> RegisterResult:
    user = db.find_by_id(user_id)
    if user is None:
        return RegisterResult(code=1919810, message="user not found")
    existing = db.find_by_name(new_name)
    if existing:
        return RegisterResult(code=1, message="username already exists")
    updated = db.update_name(user_id, new_name)
    return RegisterResult(data=updated)


# --- 测试代码 ---
class TestRegister:
    def test_register_success(self):
        db = FakeDB()
        result = register(db, "alice", "alice@test.com")
        assert result.code == 0
        assert result.data["name"] == "alice"

    def test_register_duplicate(self):
        db = FakeDB()
        register(db, "bob", "bob@test.com")
        result = register(db, "bob2", "bob@test.com")  # 同样邮箱
        assert result.code == 1
        assert "already registered" in result.message


class TestUpdateName:
    def test_update_name_success(self):
        db = FakeDB()
        register(db, "ivan", "ivan@test.com")
        user = db.find_by_name("ivan")
        result = update_name(db, user["id"], "ivan_new")
        assert result.code == 0
        assert result.data["name"] == "ivan_new"

    def test_update_name_not_found(self):
        db = FakeDB()
        result = update_name(db, 99999, "newname")
        assert result.code == 1919810

    def test_update_name_duplicate(self):
        db = FakeDB()
        register(db, "judy", "judy@test.com")
        register(db, "karl", "karl@test.com")
        user_judy = db.find_by_name("judy")
        result = update_name(db, user_judy["id"], "karl")
        assert result.code == 1


# 运行测试
print("=== 注册测试 ===")
t = TestRegister()
t.test_register_success()
print("  OK 注册成功")
t.test_register_duplicate()
print("  OK 重复邮箱拒绝")

print()
print("=== 改名测试 ===")
t2 = TestUpdateName()
t2.test_update_name_success()
print("  OK 改名成功")
t2.test_update_name_not_found()
print("  OK 用户不存在")
t2.test_update_name_duplicate()
print("  OK 名字重复拒绝")

### 服务层测试的通用模式

```
每个服务函数 -> 一个 TestClass
  +-- test_xxx_success()        <- 正常情况
  +-- test_xxx_not_found()      <- 资源不存在
  +-- test_xxx_duplicate()      <- 唯一性冲突
  +-- test_xxx_wrong_xxx()      <- 其他业务错误
```

### 项目中的错误码约定

| code | 含义 |
|------|------|
| `0` | 成功 |
| `1` | 冲突 / 未找到（登录/注册场景） |
| `2` | 密码错误 |
| `1919810` | 用户未找到（服务层其他路径） |

为什么有些用 1 有些用 1919810？这是项目内的约定，不同数字避免前端逻辑混淆，换成你自己的编号即可。

---
## 六、路由层测试 -- 用 TestClient 发 HTTP 请求

> 参考文件：`test/test_routers_user.py`

### 与服务层测试的区别

| | 服务层测试 | 路由层测试 |
|---|---|---|
| 怎么调用 | `user_service.register(session, ...)` | `client.post("/user/register", ...)` |
| 断言什么 | 函数返回值（Python 对象） | HTTP 状态码 + JSON 响应体 |
| 怎么传参 | Python 函数参数 | JSON body / URL path / query string |
| 验证的是什么 | 业务逻辑对不对 | API 接口按规范工作没有 |

In [ ]:
# ============================================================
# 路由层测试的核心：TestClient
# ============================================================

print("""
路由测试的三层断言：
  1. resp.status_code  -- HTTP 协议层（201 Created / 200 OK / 422 等）
  2. data["code"]      -- 业务状态层（0 成功 / 1 已存在 / 2 密码错误）
  3. data["data"]      -- 返回的数据内容（name、email 等）

项目中 router 测试的真实写法示例：

class TestRegisterEndpoint:
    def test_register_success(self, client):    # client 来自 conftest.py
        resp = client.post("/user/register", json={
            "name": "alice",
            "password": "pass123",
            "email": "alice@example.com",
        })

        assert resp.status_code == 201          # HTTP 层面：创建成功
        data = resp.json()                       # 解析响应

        assert data["code"] == 0                # 业务层面：操作成功
        assert data["data"]["name"] == "alice"  # 返回的数据正确
""")

### 路由测试的典型模式

```python
class TestYourEndpoint:
    def test_xxx_success(self, client):
        resp = client.post("/your/path", json={...})
        assert resp.status_code == 201
        assert resp.json()["code"] == 0

    def test_xxx_duplicate(self, client):
        client.post("/your/path", json={...})     # 第一次
        resp = client.post("/your/path", json={...}) # 第二次（冲突）
        assert resp.json()["code"] == 1

    def test_xxx_not_found(self, client):
        resp = client.get("/your/path/99999")
        assert resp.json()["code"] == 1919810
```

### 端到端联动测试（最有价值的测试模式）

```python
def test_update_password_success(self, client):
    # 步骤 1：注册
    client.post("/user/register", json={
        "name": "grace", "password": "oldpass", "email": "grace@test.com",
    })
    user_id = _get_user_id(client, "grace")

    # 步骤 2：修改密码
    resp = client.put("/user/password", json={
        "id": user_id, "old_password": "oldpass", "new_password": "newpass",
    })
    assert resp.json()["code"] == 0

    # 步骤 3：用新密码登录验证
    login_resp = client.post("/user/login", json={
        "email": "grace@test.com", "password": "newpass",
    })
    assert login_resp.json()["code"] == 0
```

### 依赖注入覆盖（怎么让路由用测试数据库？）

```python
# conftest.py 中的关键代码
app.dependency_overrides[get_session] = override_get_session
```

路由函数里所有的 `Depends(get_session)` 拿到的都是测试数据库的会话。

---
## 七、Mock：隔离外部依赖

> 参考文件：`test/test_services_comic.py`、`test/test_core_cache.py`

### 什么时候需要 Mock？

| 依赖类型 | 为什么 Mock | 示例 |
|----------|------------|------|
| 外部 API | 不可控、慢、有调用限制 | 支付接口、短信发送 |
| 数据库 | 测试要隔离、可重复 | 本项目用 SQLite 替代 MySQL |
| 文件系统 | 不污染真实文件 | 临时目录替代 |
| 网络请求 | 不可靠、慢 | 下载服务 |
| 时间 | 需要控制"现在" | TTL 过期测试 |

### unittest.mock 三件套：MagicMock、patch、side_effect

In [ ]:
# ============================================================
# unittest.mock 三件套
# ============================================================
from unittest.mock import MagicMock, patch


# ------ 1. MagicMock：一个什么都能做的"假对象" ------
print("=== MagicMock 基础 ===")

fake_db = MagicMock()
fake_db.find_by_id.return_value = {"name": "alice", "id": 1}

result = fake_db.find_by_id(1)
print(f"调用 find_by_id(1) -> {result}")

# MagicMock 记录调用情况
fake_db.find_by_id(1)
fake_db.find_by_id(2)
print(f"被调用了 {fake_db.find_by_id.call_count} 次")

# 验证调用参数
fake_db.find_by_id.assert_any_call(1)
print("assert_any_call(1) 通过")

# 验证正好调用了几次
fake_db.find_by_id.assert_called()  # 至少被调用过
print("assert_called 通过")


# ------ 2. side_effect：按调用顺序返回不同值 ------
print()
print("=== side_effect 基础 ===")

mock = MagicMock()
mock.get.side_effect = [
    None,                   # 第 1 次 -> 缓存未命中
    "/cached/path.pdf",     # 第 2 次 -> double-check 命中
]

print(f"第 1 次: {mock.get('key')}")
print(f"第 2 次: {mock.get('key')}")

# side_effect 抛异常
mock.download = MagicMock()
mock.download.side_effect = FileNotFoundError("下载失败")
try:
    mock.download(42)
except FileNotFoundError as e:
    print(f"异常抛出: {e}")

### 项目中 Mock 的真实用法

#### Mock Redis 客户端

```python
mock_client = MagicMock()
mock_client.get.return_value = "cached_value"

with patch("app.core.redis.redis_client", mock_client):
    cache = RedisCache()
    result = cache.get("key1")
    mock_client.get.assert_called_once_with("key1")
    assert result == "cached_value"
```

#### Mock 下载函数

```python
with patch("app.services.comic.download_album") as mock_download:
    with patch("pathlib.Path.exists", return_value=True):
        result = download_and_merge_pdf(1, mock_cache)

    mock_download.assert_called_once()    # 确认下载被调用
    mock_cache.set.assert_called_once()   # 确认缓存被写入
```

#### Mock 服务层（隔离路由测试）

```python
with patch(
    "app.services.comic.download_and_merge_pdf",
    return_value=mock_response,
) as mock_service:
    resp = client.get("/jm/download/1")
    mock_service.assert_called_once_with(1, mock_cache_dependency)
```

### Mock 的关键陷阱：patch 的路径

```python
# 错误：patch 的是定义处的路径
with patch("app.services.comic.some_func"):
    ...

# 正确：patch 的是使用处的路径
# router.py 里 import 之后实际调用路径是 app.routers.comic.jmcomic_service
with patch("app.routers.comic.jmcomic_service.some_func"):
    ...
```

**规则：永远 patch 在被测代码导入的路径上，而不是函数定义的路径。**

---
## 八、锁定层测试 -- 并发安全的测试技巧

> 参考文件：`test/test_core_lock.py`

### 测试并发锁需要验证什么？

1. **功能正确**：锁确实能保护临界区
2. **互斥行为**：不该有多个线程同时进入
3. **无死锁**：锁能正常释放

### 核心技巧：用共享列表记录执行顺序

In [ ]:
# ============================================================
# 锁测试模式：用共享列表验证执行顺序
# ============================================================
import threading
import time


# --- 测试模式 1：验证互斥 ---
def test_mutual_exclusion():
    lock = threading.Lock()
    active_count = []
    count_lock = threading.Lock()

    def critical_section():
        with lock:
            with count_lock:
                active_count.append(1)
            time.sleep(0.02)  # 模拟工作

    threads = [threading.Thread(target=critical_section) for _ in range(5)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()

    print(f"所有线程都执行完毕，共 {len(active_count)} 次")

test_mutual_exclusion()


# --- 测试模式 2：验证执行顺序 ---
print()
print("=== 验证执行顺序 ===")
events = []
lock2 = threading.Lock()

def worker(name, delay):
    with lock2:
        events.append(f"{name}_enter")
        time.sleep(delay)
        events.append(f"{name}_leave")

t1 = threading.Thread(target=worker, args=("A", 0.05))
t2 = threading.Thread(target=worker, args=("B", 0.02))

t1.start()
time.sleep(0.01)  # 确保 A 先拿到锁
t2.start()

t1.join()
t2.join()

print(f"执行顺序: {events}")
# 预期: ['A_enter', 'A_leave', 'B_enter', 'B_leave']
assert events == ["A_enter", "A_leave", "B_enter", "B_leave"]
print("OK 串行执行正确，B 在 A 释放后才进入")

### 锁测试的核心模式

```
1. 共享列表 events = []          <- 记录各线程进入/离开顺序
2. 启动多个线程                  <- 每个线程在持锁期间向 events 写入
3. join 等待全部完成             <- 确保所有线程执行完毕
4. 断言 events 的顺序            <- 验证锁的行为符合预期
```

### 项目中 KeyLock 的关键测试

```python
def test_same_key_serializes(self, key_lock):
    """同一 key 必须串行"""
    results = []
    def task(t):
        with key_lock.acquire(1):   # 都用 key=1
            results.append("enter")
            time.sleep(t)
            results.append("leave")
    # 启动两个都用 key=1 的任务
    # 断言: results == ["enter", "leave", "enter", "leave"]

def test_different_keys_do_not_block(self, key_lock):
    """不同 key 可以并发"""
    # 任务 A 用 key=1，任务 B 用 key=2
    # B 应该可以在 A 完成之前就完成
```

---
## 九、项目中 conftest.py 完整解读

> 参考文件：`test/conftest.py`

这是整个测试体系的"地基"，三个 fixture 逐层依赖。

In [ ]:
# ============================================================
# conftest.py 的工作原理（概念展示）
# ============================================================

print("""
=== 第一层：engine -- 内存数据库引擎 ===

engine = create_engine(
    "sqlite://",      # 空路径 = 内存数据库，测试结束自动消失
    echo=False,       # 不打印 SQL
    connect_args={"check_same_thread": False},  # FastAPI 需要跨线程
    poolclass=StaticPool,  # 单连接复用（内存数据库必须）
)
SQLModel.metadata.create_all(engine)   # [SETUP] 建表
yield engine
SQLModel.metadata.drop_all(engine)     # [CLEANUP] 删表
engine.dispose()                       # [CLEANUP] 释放


=== 第二层：session -- 数据库会话 ===

with Session(engine) as session:
    yield session
# with 退出时自动关闭 session


=== 第三层：client -- FastAPI 测试客户端 ===

# 关键 1：依赖覆盖 -- 路由用测试数据库
app.dependency_overrides[get_session] = override_get_session

# 关键 2：禁止 lifespan -- 避免连接真实 MySQL
main_module.create_db_and_tables = lambda: None

with TestClient(app) as client:
    yield client

# 关键 3：测试结束恢复原始函数
app.dependency_overrides.clear()
main_module.create_db_and_tables = original
""")

### 第一层：engine

核心知识点：
- **`sqlite://` vs `sqlite:///test.db`**：前者纯内存，进程结束后数据消失；后者持久化到文件
- **`StaticPool`**：SQLite 内存数据库必须用，否则不同 Session 看到的是不同数据库实例
- **`check_same_thread=False`**：FastAPI TestClient 内部使用多线程，SQLite 默认禁止跨线程

### 第二层：session

- 依赖 `engine` fixture，拿到独立的 Session
- 测试结束后自动关闭，未提交的修改回滚

### 第三层：client

三个关键操作：
1. **依赖覆盖**：`dependency_overrides[get_session]` -> 返回测试数据库
2. **禁止 lifespan**：`create_db_and_tables = lambda: None` -> 不连接真实 MySQL
3. **恢复原函数**：测试结束后必须恢复，否则影响其他测试

---
## 十、实战：为 MemoryCache 写测试

跟着下面的步骤，自己动手写一个完整的测试类。

### Step 1：理解被测代码

```python
# app/core/cache.py (简化版)
class MemoryCache:
    def __init__(self):
        self._store: dict[str, tuple[str, float | None]] = {}

    def set(self, key, value, ex=None):
        expire_at = time.time() + ex if ex else None
        self._store[key] = (value, expire_at)

    def get(self, key):
        if key not in self._store:
            return None
        value, expire_at = self._store[key]
        if expire_at and time.time() > expire_at:
            return None      # 已过期，惰性删除
        return value

    def delete(self, key):
        self._store.pop(key, None)

    def exists(self, key):
        return self.get(key) is not None
```

### Step 2：列出所有需要测试的场景

```
set + get -> 值正确
get(不存在) -> None
set 覆写 -> 新值覆盖旧值
delete -> 取不到
exists -> True/False
TTL 未过期 -> 正常取值
TTL 已过期 -> 返回 None
delete 不存在 key -> 不报错
```

### Step 3：写测试代码

In [ ]:
# ============================================================
# 实战：为 MemoryCache 写测试
# ============================================================
import time
import pytest


# --- 被测代码 ---
class MemoryCache:
    def __init__(self):
        self._store: dict[str, tuple[str, float | None]] = {}

    def set(self, key: str, value: str, ex: int | None = None):
        expire_at = time.time() + ex if ex else None
        self._store[key] = (value, expire_at)

    def get(self, key: str) -> str | None:
        if key not in self._store:
            return None
        value, expire_at = self._store[key]
        if expire_at and time.time() > expire_at:
            return None
        return value

    def delete(self, key: str):
        self._store.pop(key, None)

    def exists(self, key: str) -> bool:
        return self.get(key) is not None


# --- 测试代码 ---
class TestMemoryCache:
    @pytest.fixture
    def cache(self):
        return MemoryCache()

    def test_set_and_get(self, cache):
        cache.set("user:1", "Alice")
        assert cache.get("user:1") == "Alice"

    def test_get_missing(self, cache):
        assert cache.get("nonexistent") is None

    def test_set_overwrite(self, cache):
        cache.set("key", "old")
        cache.set("key", "new")
        assert cache.get("key") == "new"

    def test_delete(self, cache):
        cache.set("key", "value")
        cache.delete("key")
        assert cache.get("key") is None

    def test_exists(self, cache):
        cache.set("key", "value")
        assert cache.exists("key") is True
        assert cache.exists("ghost") is False

    def test_ttl_not_expired(self, cache):
        cache.set("key", "value", ex=3600)
        assert cache.get("key") == "value"

    def test_ttl_expired(self, cache):
        now = time.time()
        cache.set("key", "value", ex=60)
        # 手动把过期时间改到过去，证明过期逻辑正确
        cache._store["key"] = (cache._store["key"][0], now - 1)
        assert cache.get("key") is None

    def test_delete_nonexistent(self, cache):
        cache.delete("ghost")  # 不抛异常即通过


# --- 手动运行验证 ---
tester = TestMemoryCache()
cache = MemoryCache()

cache.set("user:1", "Alice")
assert cache.get("user:1") == "Alice"
print("OK test_set_and_get")

assert cache.get("nope") is None
print("OK test_get_missing")

cache.set("k", "v1"); cache.set("k", "v2")
assert cache.get("k") == "v2"
print("OK test_set_overwrite")

cache.set("temp", "val"); cache.delete("temp")
assert cache.get("temp") is None
print("OK test_delete")

cache.set("exists_test", "y")
assert cache.exists("exists_test") is True
assert cache.exists("nope") is False
print("OK test_exists")

cache.set("ttl_test", "v", ex=3600)
assert cache.get("ttl_test") == "v"
print("OK test_ttl_not_expired")

now = time.time()
cache.set("ttl_exp", "v", ex=60)
cache._store["ttl_exp"] = (cache._store["ttl_exp"][0], now - 1)
assert cache.get("ttl_exp") is None
print("OK test_ttl_expired")

cache.delete("ghost")
print("OK test_delete_nonexistent")

print()
print("全部 8 个测试通过！")

---

## 总结

### 三层测试架构

```
+---------------------------------------------------------------+
|  路由层测试 (test_routers_*.py)                                  |
|  工具: TestClient                                              |
|  验证: HTTP 状态码、JSON 响应、端到端流程                          |
+---------------------------------------------------------------+
|  服务层测试 (test_services_*.py)                                 |
|  工具: session fixture                                          |
|  验证: 业务逻辑、错误码、数据变更                                   |
+---------------------------------------------------------------+
|  模型层测试 (test_models.py)                                     |
|  工具: 无依赖，直接实例化                                         |
|  验证: 字段定义、默认值、校验规则                                   |
+---------------------------------------------------------------+
```

### 测试设计五原则

| 原则 | 说明 |
|------|------|
| **单一职责** | 每个测试只验证一件事 |
| **正向+反向** | 每个功能都测"成功"和"失败"两条路径 |
| **命名即文档** | `test_register_duplicate_email` 一看就懂 |
| **测试独立** | 每个测试不依赖其他测试的执行顺序 |
| **隔离外部** | 用 fixture + mock 隔离数据库/网络/文件 |

### 进阶方向

本笔记覆盖了项目中所有测试模式。更深入的测试技能包括：

- **集成测试**：测试多个模块一起工作
- **端到端测试**：模拟真实用户操作全流程
- **覆盖率工具**：`pytest --cov` 检查哪些代码没被测试
- **CI 集成**：在 GitHub Actions 中自动运行测试
- **TDD**：先写测试再写代码的开发方式